# Test Scala Data Provenance

### Prerequisites

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

### Import libraries

In [2]:
val packageVersion = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % packageVersion)  // use programmatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

packageVersion: String = "0.0.1"

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

import org.dataprov.dp.wringlet.ProvenanceApi._
import org.dataprov.dp.wringlet.SparkProvenanceExtension
import org.dataprov.dp.wringlet.LogicalPlanWithProvenance


import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._
import org.dataprov.dp.wringlet.ProvenanceApi._
import org.dataprov.dp.wringlet.SparkProvenanceExtension
import org.dataprov.dp.wringlet.LogicalPlanWithProvenance

### Initialize spark session

In [4]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .withExtensions(new SparkProvenanceExtension())
  // .config("spark.jars", s"../target/scala-2.13/dp-spark_2.13-$packageVersion.jar")
  // .config("spark.sql.extensions", "org.dataprov.dp.ProvenanceExtension")
  .config("spark.provenance.enabled", "true")
  .getOrCreate()


println(s"Spark provenance enabled: ${spark.conf.get("spark.provenance.enabled")}")
  

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/10 15:17:47 INFO SparkContext: Running Spark version 4.1.1
26/07/10 15:17:47 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/07/10 15:17:47 INFO SparkContext: Java version 17.0.10+7
26/07/10 15:17:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/10 15:17:47 INFO ResourceUtils: ==============================================================
26/07/10 15:17:47 INFO ResourceUtils: No custom resources configured for spark.driver.
26/07/10 15:17:47 INFO ResourceUtils: ==============================================================
26/07/10 15:17:47 INFO SparkContext: Submitted application: notebook
26/07/10 15:17:47 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/07/10 15:17:47 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/07/10 15:17:47 INFO SecurityManager: Changing view acls groups to

Spark provenance enabled: true


spark: SparkSession = org.apache.spark.sql.classic.SparkSession@23bcce82

### Create spark dataframe

We create a dataframe to test the provenance annotations.

In [5]:
val df: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+



df: DataFrame = [A: string, B: string ... 1 more field]

The Provenance column is added to the dataframe

In [6]:
val dfWithProvenance : DataFrame = df.addProvenanceColumn
dfWithProvenance.printSchema()
dfWithProvenance.show(false)

root
 |-- A: string (nullable = true)
 |-- B: string (nullable = true)
 |-- C: string (nullable = true)
 |-- _provenance_tag: string (nullable = false)

+---+---+---+------------------------------------+
|A  |B  |C  |_provenance_tag                     |
+---+---+---+------------------------------------+
|a  |b  |c  |1d271268-e577-4593-8d74-c30152dae57b|
|d  |b  |e  |45cc84c4-b224-42ec-b5ef-b35501fc6207|
|f  |g  |e  |baaa3e73-5369-46a4-9fab-6fed00ffe1f8|
+---+---+---+------------------------------------+



dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

### Tests for join, with and without provenance

With dataframes

In [7]:
val df2 : DataFrame = df
    .select("A", "B")
    .join(df.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2.show()

val df2WithProvenance : DataFrame = dfWithProvenance
    .select("A", "B")
    .join(dfWithProvenance.select("B", "C"), "B")
    .select("A", "B", "C")
    .orderBy("A", "B", "C")

df2WithProvenance.show(false)


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |_provenance_tag                                                              |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|a  |b  |e  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|d  |b  |c  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|d  |b  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|f  |g  |e  |(baaa3e73-5369-46a4-9fab-6fed00ffe1f8 ⊗ baaa3e73-5369-46a4-9fab-6fed00ffe1f8)|
+---+---+---+-----------------------------------------------------------------------------+



df2: DataFrame = [A: string, B: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

With views

In [8]:
df.createOrReplaceTempView("df_without_prov")

val df3 = spark.sql("""
    SELECT A, B, t1.C
    FROM (
        SELECT A, C
        FROM df_without_prov
    ) AS t1
    JOIN (
        SELECT B, C
        FROM df_without_prov
    ) AS t2
    ON t1.C = t2.C
    ORDER BY A, B, t1.C
""")
df3.show()

dfWithProvenance.createOrReplaceTempView("df_with_prov")
val df3WithProvenance: DataFrame = spark.sql("""
    SELECT A, B, t1.C
    FROM (
        SELECT A, C
        FROM df_with_prov
    ) AS t1
    JOIN (
        SELECT B, C
        FROM df_with_prov
    ) AS t2
    ON t1.C = t2.C
    ORDER BY A, B, t1.C
""")

df3WithProvenance.show(false)


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |_provenance_tag                                                              |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|d  |b  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|d  |g  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ baaa3e73-5369-46a4-9fab-6fed00ffe1f8)|
|f  |b  |e  |(baaa3e73-5369-46a4-9fab-6fed00ffe1f8 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|f  |g  |e  |(baaa3e73-5369-46a4-9fab-6fed00ffe1f8 ⊗ baaa3e73-5369-46a4-9fab-6fed00ffe1f8)|
+---+---+---+-----------------------------------------------------------------------------+



df3: DataFrame = [A: string, B: string ... 1 more field]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

### Tests for union

In [9]:
val df4 = df2.union(df3).distinct().orderBy("A", "B", "C")
df4.show()

val df4WithProvenance = df2WithProvenance.union(df3WithProvenance).distinct().orderBy("A", "B", "C")
df4WithProvenance.show(false)

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |_provenance_tag                                                              |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|a  |b  |e  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|d  |b  |c  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|d  |b  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|d  |g  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ baaa3e73-5369-46a4-9fab-6fed00ffe1f8)|
|f  |b  |e  |(baaa3e73-5369-46a4-9fab-6fed00ffe1f8 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|f  |g  |e  |(baa

df4: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 1 more field]
df4WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

In [10]:
df2.createOrReplaceTempView("df2_without_prov")
df3.createOrReplaceTempView("df3_without_prov")
val df4_bis = spark.sql("""
    SELECT DISTINCT A, B, C
    FROM (
        SELECT A, B, C
        FROM df2_without_prov
        UNION
        SELECT A, B, C
        FROM df3_without_prov
    )
    ORDER BY A, B, C
""")
df4_bis.show()

df2WithProvenance.createOrReplaceTempView("df2_with_prov")
df3WithProvenance.createOrReplaceTempView("df3_with_prov")
val df4WithProvenance_bis = spark.sql("""
    SELECT DISTINCT A, B, C
    FROM (
        SELECT A, B, C
        FROM df2_with_prov
        UNION
        SELECT A, B, C
        FROM df3_with_prov
    )
    ORDER BY A, B, C
""")
df4WithProvenance_bis.show(false)

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  e|
|  d|  g|  e|
|  f|  b|  e|
|  f|  g|  e|
+---+---+---+

+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |_provenance_tag                                                              |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|a  |b  |e  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|d  |b  |c  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 1d271268-e577-4593-8d74-c30152dae57b)|
|d  |b  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|d  |g  |e  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ baaa3e73-5369-46a4-9fab-6fed00ffe1f8)|
|f  |b  |e  |(baaa3e73-5369-46a4-9fab-6fed00ffe1f8 ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)|
|f  |g  |e  |(baa

df4_bis: DataFrame = [A: string, B: string ... 1 more field]
df4WithProvenance_bis: DataFrame = [A: string, B: string ... 2 more fields]

### Tests for distinct

In [11]:
val df5 = df4.select("A", "C").distinct().orderBy("A", "C")
df5.show()

val df5WithProvenance = df4WithProvenance.select("A", "C").distinct().orderBy("A", "C")
df5WithProvenance.show(false)

+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|A  |C  |_provenance_tag                                                                                                                                                |
+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|a  |c  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b)                                                                                  |
|a  |e  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)                                                                                  |
|d  |c  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 1d271268-e

df5: Dataset[org.apache.spark.sql.Row] = [A: string, C: string]
df5WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, C: string ... 1 more field]

In [12]:
df4.createOrReplaceTempView("df4_without_prov")
val df5_bis = spark.sql("""
    SELECT DISTINCT A, C
    FROM df4_without_prov
    ORDER BY A, C
""")
df5_bis.show()

df4WithProvenance.createOrReplaceTempView("df4_with_prov")
val df5WithProvenance = spark.sql("""
    SELECT DISTINCT A, C
    FROM df4_with_prov
    ORDER BY A, C
""")
df5WithProvenance.show(false)


+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|A  |C  |_provenance_tag                                                                                                                                                |
+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|a  |c  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b)                                                                                  |
|a  |e  |(1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207)                                                                                  |
|d  |c  |(45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 1d271268-e

df5_bis: DataFrame = [A: string, C: string]
df5WithProvenance: DataFrame = [A: string, C: string ... 1 more field]

### Tests for filter and sort

In [13]:
val df6WithProvenance = dfWithProvenance.sort("A", "B", "C").filter("A = 'a' OR A = 'f'")
df6WithProvenance.show(false)


+---+---+---+------------------------------------+
|A  |B  |C  |_provenance_tag                     |
+---+---+---+------------------------------------+
|a  |b  |c  |1d271268-e577-4593-8d74-c30152dae57b|
|f  |g  |e  |baaa3e73-5369-46a4-9fab-6fed00ffe1f8|
+---+---+---+------------------------------------+



df6WithProvenance: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

### Few more tests for joins

Test for multiple joins in a single query

In [14]:
val df7WithProvenance: DataFrame = df2WithProvenance
    .select(col("A"), col("B").as("B1"), col("C").as("C1"))
    .join(
        df3WithProvenance.select(col("A"), col("B").as("B2"), col("C").as("C2")),
        Seq("A")
    )
    .select("A", "B1", "C1", "B2", "C2")

df7WithProvenance.show(false)

+---+---+---+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|A  |B1 |C1 |B2 |C2 |_provenance_tag                                                                                                                                                |
+---+---+---+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|a  |b  |e  |b  |c  |((1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207) ⊗ (1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b))|
|a  |b  |c  |b  |c  |((1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b) ⊗ (1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b))|
|d  |b  |e  |b  |e  |((45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 45cc84c4-b224-42ec-b5ef-b355

df7WithProvenance: DataFrame = [A: string, B1: string ... 4 more fields]

In [15]:
df2WithProvenance.createOrReplaceTempView("df2_with_prov")
df3WithProvenance.createOrReplaceTempView("df3_with_prov")
val df7WithProvenance : DataFrame = spark.sql("""
    SELECT t1.A, B1, C1, B2, C2
    FROM (
        SELECT A, B as B1, C as C1
        FROM df2_with_prov
    ) AS t1
    JOIN (
        SELECT A, B as B2, C as C2
        FROM df3_with_prov
    ) AS t2
    ON t1.A = t2.A
""")

df7WithProvenance.show(false)

+---+---+---+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|A  |B1 |C1 |B2 |C2 |_provenance_tag                                                                                                                                                |
+---+---+---+---+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------+
|a  |b  |e  |b  |c  |((1d271268-e577-4593-8d74-c30152dae57b ⊗ 45cc84c4-b224-42ec-b5ef-b35501fc6207) ⊗ (1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b))|
|a  |b  |c  |b  |c  |((1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b) ⊗ (1d271268-e577-4593-8d74-c30152dae57b ⊗ 1d271268-e577-4593-8d74-c30152dae57b))|
|d  |b  |e  |b  |e  |((45cc84c4-b224-42ec-b5ef-b35501fc6207 ⊗ 45cc84c4-b224-42ec-b5ef-b355

df7WithProvenance: DataFrame = [A: string, B1: string ... 4 more fields]

In [16]:
val dfDistinct = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("a", "b", "c"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C").addProvenanceColumn

dfDistinct.show(false)

val test = dfDistinct.select("A", "B", "C").distinct()
test.show(false)

+---+---+---+------------------------------------+
|A  |B  |C  |_provenance_tag                     |
+---+---+---+------------------------------------+
|a  |b  |c  |19bae440-e16b-46b6-9b56-5b35395b88f6|
|a  |b  |c  |8a4dd0b7-e477-4a0f-81b1-bdd2443d69a5|
|f  |g  |e  |37bfa7af-9203-4401-8358-bd2b9cba370d|
+---+---+---+------------------------------------+

+---+---+---+-----------------------------------------------------------------------------+
|A  |B  |C  |_provenance_tag                                                              |
+---+---+---+-----------------------------------------------------------------------------+
|a  |b  |c  |{19bae440-e16b-46b6-9b56-5b35395b88f6 ⊕ 8a4dd0b7-e477-4a0f-81b1-bdd2443d69a5}|
|f  |g  |e  |37bfa7af-9203-4401-8358-bd2b9cba370d                                         |
+---+---+---+-----------------------------------------------------------------------------+



dfDistinct: DataFrame = [A: string, B: string ... 2 more fields]
test: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

**Another dataframe to test joins**

In [17]:

import java.sql.Date

val df = spark.createDataFrame(
    Seq(
    ("A", Date.valueOf("2026-01-15"), 10.0, 90),
    ("A", Date.valueOf("2026-01-16"), 10.0, 120),
    ("A", Date.valueOf("2026-01-17"), 5.0, 300),
    ("B", Date.valueOf("2026-01-15"), 100.0, 20),
    ("B", Date.valueOf("2026-01-16"), 100.0, 30),
    ("C", Date.valueOf("2026-01-17"), 80.0, 60),
    ("F", Date.valueOf("2026-01-16"), 50.0, 70)
)).toDF("product", "date", "price", "quantity")

val df_prov = df.addProvenanceColumn
df_prov.show(false)

+-------+----------+-----+--------+------------------------------------+
|product|date      |price|quantity|_provenance_tag                     |
+-------+----------+-----+--------+------------------------------------+
|A      |2026-01-15|10.0 |90      |defd4324-aed7-4503-96b8-8a64e98226cd|
|A      |2026-01-16|10.0 |120     |bcfbb061-13c1-4def-b953-4a9a5285bbf4|
|A      |2026-01-17|5.0  |300     |80e2bee6-add2-4ec0-8d88-765338349f94|
|B      |2026-01-15|100.0|20      |afd2b3ed-bdc0-47ad-9cc7-16fd6e5b7702|
|B      |2026-01-16|100.0|30      |e93f9837-3ea9-4222-842b-ed23819063fc|
|C      |2026-01-17|80.0 |60      |d0b2a9ef-f19d-4364-9b4a-c9e3adc27a68|
|F      |2026-01-16|50.0 |70      |49d46b4a-f45e-486a-8374-38fae3b0d14c|
+-------+----------+-----+--------+------------------------------------+



import java.sql.Date
df: DataFrame = [product: string, date: date ... 2 more fields]
df_prov: DataFrame = [product: string, date: date ... 3 more fields]

**3 examples with same result expected**

The following tests check that distinct and dropDuplicates on a column with provenance work correctly and do not cause any errors. 

The expected output is that the distinct products are returned without any errors, the provenance tags are correctly aggregated for duplicate rows.

We can observe the differences between the Parsed Logical Plan, the Analysed Logical Plan and the Optimized Logical Plan.

In [18]:
df_prov.createOrReplaceTempView("table_with_prov")
val test = spark.sql("SELECT distinct product FROM table_with_prov")
test.show(false)

val test2 = df_prov.select("product").dropDuplicates("product")
test2.show(false)

val test3 = df_prov.select("product").distinct()
test3.show(false)

+-------+--------------------------------------------------------------------------------------------------------------------+
|product|_provenance_tag                                                                                                     |
+-------+--------------------------------------------------------------------------------------------------------------------+
|A      |{defd4324-aed7-4503-96b8-8a64e98226cd ⊕ bcfbb061-13c1-4def-b953-4a9a5285bbf4 ⊕ 80e2bee6-add2-4ec0-8d88-765338349f94}|
|B      |{afd2b3ed-bdc0-47ad-9cc7-16fd6e5b7702 ⊕ e93f9837-3ea9-4222-842b-ed23819063fc}                                       |
|C      |d0b2a9ef-f19d-4364-9b4a-c9e3adc27a68                                                                                |
|F      |49d46b4a-f45e-486a-8374-38fae3b0d14c                                                                                |
+-------+------------------------------------------------------------------------------------------------------

test: DataFrame = [product: string, _provenance_tag: string]
test2: Dataset[org.apache.spark.sql.Row] = [product: string, _provenance_tag: string]
test3: Dataset[org.apache.spark.sql.Row] = [product: string, _provenance_tag: string]

In [19]:
df.createOrReplaceTempView("table_without_prov")
val test4 = spark.sql("SELECT distinct product FROM table_without_prov")
test4.show()

val test5 = df.select("product").distinct()
test5.show()

val test6 = df.select("product").dropDuplicates("product")
test6.show()

+-------+
|product|
+-------+
|      A|
|      B|
|      C|
|      F|
+-------+

+-------+
|product|
+-------+
|      A|
|      B|
|      C|
|      F|
+-------+

+-------+
|product|
+-------+
|      A|
|      B|
|      C|
|      F|
+-------+



test4: DataFrame = [product: string]
test5: Dataset[org.apache.spark.sql.Row] = [product: string]
test6: Dataset[org.apache.spark.sql.Row] = [product: string]

In [20]:
val test7 = df_prov
    .select("product", "price")
    .withColumns(Map(
        "price2" -> (col("price") + lit(2)),
        "price3" -> (col("price") + lit(3))
    ))
test7.explain(true)
test7.show(false)

val test8 = test7.select("product", "price2")
test8.explain(true)
test8.show(false)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(price2, price3, '`+`('price, 2), '`+`('price, 3), None)]
+- Project [product#541, price#543, _provenance_tag#545 AS _provenance_tag#709]
   +- Project [product#541, date#542, price#543, quantity#544, uuid(Some(-7862727232339943892)) AS _provenance_tag#545]
      +- Project [_1#537 AS product#541, _2#538 AS date#542, _3#539 AS price#543, _4#540 AS quantity#544]
         +- LocalRelation [_1#537, _2#538, _3#539, _4#540]

== Analyzed Logical Plan ==
product: string, price: double, price2: double, price3: double, _provenance_tag: string
Project [product#541, price#543, (price#543 + cast(2 as double)) AS price2#710, (price#543 + cast(3 as double)) AS price3#711, _provenance_tag#709]
+- Project [product#541, price#543, _provenance_tag#545 AS _provenance_tag#709]
   +- Project [product#541, date#542, price#543, quantity#544, uuid(Some(-7862727232339943892)) AS _provenance_tag#545]
      +- Project [_1#537 AS product#541, _2#538 AS 

test7: DataFrame = [product: string, price: double ... 3 more fields]
test8: DataFrame = [product: string, price2: double ... 1 more field]